# Volume 10, Chapter 4: The central limit theorem, sharpened (Python companion)

Exercises first, solutions at the end. Exact Berry–Esseen errors, the cost of insertion sort, Weldon's dice and Pearson's test, the local limit theorem and what goes wrong on a sublattice, Pólya's constant from a Bessel integral, and a computer science application: **testing hash functions with the $\chi^2$ test**. Requires `numpy` and `scipy`.

In [ ]:
import math
import numpy as np
from scipy import stats, special, integrate
rng = np.random.default_rng(40)

## Exercises

**P4.1 (★) Exact Berry–Esseen errors.** For $S_n$ binomial$(n,p)$, compute $D_n=\sup_x|P((S_n-np)/\sqrt{npq}\le x)-\Phi(x)|$ exactly (the supremum is attained at an atom, from the left or from the right). For $p=\frac12$ and $p=0.01$, find the smallest $n$ with $D_n<0.01$, and compare with what Theorem 4.7 (constant $3.1$) and Shevtsova's constant $0.4748$ guarantee.

**P4.2 (★) Inversions and insertion sort.** Count the inversions of $2000$ random permutations of size $n=5000$ with an $O(n\log n)$ merge-sort count, and compare the mean, the standard deviation and the histogram of $(I_n-n^2/4)/(n^{3/2}/6)$ with Example 4.3. Then time insertion sort on random data of a few sizes: is its running time proportional to the number of inversions?

**P4.3 (★★) Weldon's dice.** (a) Compute Pearson's statistic for Weldon's data (Example 4.10) under $p=\frac13$ with all thirteen cells, and its $p$-value under $\chi^2_{12}$. (b) Two cells have expected counts near $1$ and $0.05$, against the usual rule of thumb "at least $5$ per cell". Check by simulation (fair dice, $26\,306$ throws, $2\times10^4$ repetitions) whether the $\chi^2_{12}$ approximation is trustworthy. Then merge the cells $10,11,12$ and redo (a) and the simulation with $10$ degrees of freedom. (c) Estimate $p$ from the data and recompute the merged statistic, with $9$ degrees of freedom.

**P4.4 (★★) Span.** (a) For steps uniform on $\{0,1,3\}$, compare $\sigma\sqrt n\,P(S_n=k)$ with $\phi((k-n\mu)/(\sigma\sqrt n))$ for $n=10,100,1000$ (exact convolution), and record the maximal error. (b) Repeat with steps uniform on $\{0,2,6\}$, which has span $2$. What goes wrong, and what is the right statement?

**P4.5 (★★) Pólya's constant.** Using $\frac1{1-\varphi}=\int_0^\infty e^{-s(1-\varphi)}\dd s$ and $\frac1{2\pi}\int_{-\pi}^\pi e^{s\cos\theta}\dd\theta=I_0(s)$ (a Bessel function), show that for the simple walk on $\Z^d$, $\sum_{n\ge0}u_n=\int_0^\infty\bigl(e^{-s/d}I_0(s/d)\bigr)^d\dd s$. Compute it for $d=3$ (Watson: $1.5164$), deduce the return probability, and check it by simulation. What happens for $d=2$?

**P4.6 (★★★) CS application: is your hash function uniform?** A hash table with $m$ buckets stores keys $k$ in bucket $h(k)$. If $n$ keys hash "at random", the bucket counts are multinomial and Pearson's statistic is approximately $\chi^2_{m-1}$. Take $m=1024$ and $n=10^5$ keys that are multiples of $64$ (addresses of aligned objects, say). Compare (a) $h(k)=k\bmod m$, (b) $h(k)=k\bmod1021$ (a prime number of buckets), (c) Knuth's multiplicative hashing $h(k)=\lfloor m\{kA\}\rfloor$ with $A=(\sqrt5-1)/2$, (d) Python's built-in `hash` on the strings `str(k)`, with `PYTHONHASHSEED` effects in mind. Report $\chi^2$, the $p$-value, and the longest chain. Why does the test catch (a)? Could a hash pass the $\chi^2$ test and still be bad?

## Solutions

### P4.1
**How to find it.** Between atoms, $F_n$ is constant and $\Phi$ is monotone, so the largest gap is at an atom: compare $\Phi(x_k)$ with $F_n(x_k)$ and with $F_n(x_k-)=F_n(x_k)-P(S_n=k)$. For Bernoulli$(p)$, $\rho/\sigma^3=(p^2+q^2)/\sqrt{pq}$.

In [ ]:
def kolmogorov_binom(n, p):
    q = 1 - p
    k = np.arange(n + 1)
    x = (k - n * p) / math.sqrt(n * p * q)
    F = stats.binom.cdf(k, n, p)
    Fm = F - stats.binom.pmf(k, n, p)
    P = stats.norm.cdf(x)
    return max(np.max(np.abs(F - P)), np.max(np.abs(Fm - P)))

for p in (0.5, 0.01):
    q = 1 - p
    ratio = (p * p + q * q) / math.sqrt(p * q)
    n = 1
    while kolmogorov_binom(n, p) >= 0.01:
        n = n + 1 if n < 100 else int(n * 1.02) + 1
    lo = max(1, int(n / 1.02) - 2)
    n = next(m for m in range(lo, n + 1) if kolmogorov_binom(m, p) < 0.01)
    print(f"p={p}: D_n < 0.01 from n = {n} (D_n = {kolmogorov_binom(n, p):.5f}); "
          f"guaranteed by C=3.1 for n >= {math.ceil((3.1 * ratio / 0.01)**2)}, "
          f"by C=0.4748 for n >= {math.ceil((0.4748 * ratio / 0.01)**2)}")

For $p=\frac12$ the truth (about $1600$ throws; note that $D_n$ is not monotone, so the search stops at the first success) and Shevtsova's guarantee ($2255$) are close: the coin is nearly the extremal case for the constant. For $p=0.01$ the guaranteed sizes are far larger, because $\rho/\sigma^3\approx10$ is large; the true error is also slow to decrease, because the law of rare successes is close to Poisson, not Gaussian, until $np$ is large. Our constant $3.1$ asks for about $60$ times more data than needed: a proof constant is a guarantee, not an estimate.

### P4.2
**How to find it.** Merge sort counts inversions as it merges: when an element of the right half is placed before the remaining $r$ elements of the left half, it accounts for $r$ inversions. Vectorising over permutations is awkward, so use `numpy`'s stable sort on the inversion table instead: by Example 4.3, $I_n=\sum_kY_k$ with independent uniform $Y_k$, but to *check* the example we must count on actual permutations. A Fenwick tree (binary indexed tree) does it in $O(n\log n)$ too.

In [ ]:
def inversions(a):
    # merge sort count, iterative bottom-up, O(n log n)
    a = list(a); n = len(a); count = 0; width = 1
    buf = [0] * n
    while width < n:
        for lo in range(0, n, 2 * width):
            mid, hi = min(lo + width, n), min(lo + 2 * width, n)
            i, j, k = lo, mid, lo
            while i < mid and j < hi:
                if a[i] <= a[j]:
                    buf[k] = a[i]; i += 1
                else:
                    buf[k] = a[j]; j += 1; count += mid - i
                k += 1
            buf[k:hi] = a[i:mid] + a[j:hi]
        a, buf = buf, a
        width *= 2
    return count

n, reps = 5000, 300
I = np.array([inversions(rng.permutation(n)) for _ in range(reps)])
s = math.sqrt((2 * n**3 + 3 * n**2 - 5 * n) / 72)
print(f"mean {I.mean():.0f} (theory {n*(n-1)/4:.0f}),  sd {I.std():.0f} (theory {s:.0f})")
Z = (I - n * (n - 1) / 4) / s
print("Kolmogorov distance of the standardised counts to N(0,1):", round(stats.kstest(Z, 'norm').statistic, 3))

In [ ]:
import time
def insertion_sort(a):
    a = list(a)
    for i in range(1, len(a)):
        x = a[i]; j = i - 1
        while j >= 0 and a[j] > x:
            a[j + 1] = a[j]; j -= 1
        a[j + 1] = x
    return a

for n in (500, 1000, 2000):
    p = rng.permutation(n)
    t0 = time.perf_counter(); insertion_sort(p); t = time.perf_counter() - t0
    print(f"n={n}: inversions {inversions(p):>8d}, time {t*1e3:7.1f} ms, ns per inversion {t/inversions(p)*1e9:.0f}")

(I used $300$ permutations rather than $2000$ to keep the pure-Python count fast; the conclusions do not change.) The mean and the standard deviation match Example 4.3, and the Kolmogorov distance is of the size expected from $300$ samples. The time per inversion is roughly constant: insertion sort's cost on random input *is* its number of inversions, about $n^2/4$, and the central limit theorem says that it almost never deviates from that by more than a few multiples of $n^{3/2}/6$, a vanishing fraction.

### P4.3
**How to find it.** Expected counts $26\,306\binom{12}k3^{-k}(2/3)^{12-k}$; the estimate of $p$ is the total number of fives and sixes divided by $12\times26\,306$. For (c), simulate multinomial vectors with the fair-dice probabilities.

In [ ]:
obs = np.array([185, 1149, 3265, 5475, 6114, 5194, 3067, 1331, 403, 105, 14, 4, 0])
N = obs.sum(); k = np.arange(13)
def pearson(obs, probs):
    e = obs.sum() * probs
    return np.sum((obs - e)**2 / e)
p0 = stats.binom.pmf(k, 12, 1/3)
x0 = pearson(obs, p0)
print(f"(a) N = {N}, chi2 = {x0:.2f}, p-value (12 df) = {stats.chi2.sf(x0, 12):.2e}")
sims = rng.multinomial(N, p0, size=20000)
full = np.sum((sims - N * p0)**2 / (N * p0), axis=1)
print(f"(b) 13 cells: simulated P(chi2 >= {x0:.1f}) = {np.mean(full >= x0):.4f}, chi2_12 says {stats.chi2.sf(x0, 12):.5f};"
      f" simulated 99% quantile {np.quantile(full, 0.99):.1f} vs {stats.chi2.ppf(0.99, 12):.1f}")
merge = lambda v: np.concatenate([v[..., :10], v[..., 10:].sum(axis=-1, keepdims=True)], axis=-1)
pm = merge(p0); xm = pearson(merge(obs), pm)
simm = np.sum((merge(sims) - N * pm)**2 / (N * pm), axis=1)
print(f"    11 cells: chi2 = {xm:.2f}, p-value (10 df) = {stats.chi2.sf(xm, 10):.1e}, simulated {np.mean(simm >= xm):.5f};"
      f" simulated 99% quantile {np.quantile(simm, 0.99):.1f} vs {stats.chi2.ppf(0.99, 10):.1f}")
phat = (obs * k).sum() / (12 * N)
x1 = pearson(merge(obs), merge(stats.binom.pmf(k, 12, phat)))
print(f"(c) p_hat = {phat:.4f}, merged chi2 = {x1:.2f}, p-value (9 df) = {stats.chi2.sf(x1, 9):.3f}")

(a) $41.3$, and $\chi^2_{12}$ would give $p\approx4\times10^{-5}$. (b) But the simulation shows that with the thirteen cells the statistic has a much heavier tail than $\chi^2_{12}$: the empty last cell, expecting $0.05$ throws, adds about $20$ whenever it receives a single throw, which happens about one time in twenty. The true $p$-value of $41.3$ is about two in a thousand: still small, but some fifty times larger than the $\chi^2$ table says. After merging, the approximation is excellent (compare the quantiles), and the merged statistic $35.5$ has $p\approx10^{-4}$, consistent with the simulation (which should see about two exceedances in $2\times10^4$ runs, and sees none or a handful). Pearson's conclusion stands; his number did not. (c) With $\hat p\approx0.3377$ the merged statistic is $8.2$ with $p\approx0.5$, the values found by Labby in 2009 after correcting Pearson's arithmetic: biased dice fit perfectly.

### P4.4
**How to find it.** Convolve the step law $n$ times with `np.convolve`. For span $2$, $S_n$ lives on even integers, so $P(S_n=k)$ is $0$ on odd $k$ and about *twice* the Gaussian prediction on even $k$.

In [ ]:
def llt_error(values, n):
    step = np.zeros(max(values) + 1)
    for v in values: step[v] += 1 / len(values)
    mu = sum(values) / len(values); var = sum(v * v for v in values) / len(values) - mu**2
    d = np.array([1.0])
    for _ in range(n): d = np.convolve(d, step)
    kk = np.arange(len(d)); s = math.sqrt(n * var)
    pred = stats.norm.pdf((kk - n * mu) / s)
    return np.max(np.abs(s * d - pred)), d, pred, s

for values in ([0, 1, 3], [0, 2, 6]):
    print(values, [round(llt_error(values, n)[0], 4) for n in (10, 100, 1000)])
err, d, pred, s = llt_error([0, 2, 6], 100)
centre = int(np.argmax(d))
print("span 2, n=100, around the mode: s*P(S=k) =", np.round(s * d[centre - 2:centre + 3], 3), " prediction:", np.round(pred[centre - 2:centre + 3], 3))

For span $1$ the error tends to $0$ like $n^{-1/2}$. For span $2$ it does not tend to $0$ at all: the mass sits on even integers, so it is $0$ on odd $k$ and twice the Gaussian prediction on even $k$. The right statement for span $h$: if $X\in x_0+h\Z$, then $\sigma\sqrt n\,P(S_n=k)=h\,\phi\bigl(\frac{k-n\mu}{\sigma\sqrt n}\bigr)+o(1)$ uniformly over $k\in nx_0+h\Z$ (apply Theorem 4.12 to $(X-x_0)/h$).

### P4.5
**How to find it.** For the simple walk $\varphi(t)=\frac1d\sum_j\cos t_j$, so $e^{-s(1-\varphi(t))}=e^{-s}\prod_je^{(s/d)\cos t_j}$, and integrating each coordinate gives $I_0(s/d)$. So $(2\pi)^{-d}\int\frac{\dd t}{1-\varphi}=\int_0^\infty e^{-s}I_0(s/d)^d\dd s$ (Tonelli: everything is non-negative). `scipy.special.ive(0, x)` is $e^{-x}I_0(x)$, numerically stable.

In [ ]:
def green(d, upper=np.inf):
    f = lambda s: special.ive(0, s / d)**d
    return integrate.quad(f, 0, 50, limit=500)[0] + integrate.quad(f, 50, upper, limit=500)[0]

G3 = green(3)
print(f"d=3: sum u_n = {G3:.6f}, return probability F = 1 - 1/G = {1 - 1/G3:.6f}")
for U in (1e2, 1e4, 1e6):
    print(f"d=2: integral up to s={U:.0e}: {green(2, U):.3f}")

# simulation of returns in Z^3, within a time horizon
walks, horizon = 20000, 2000
steps = np.array([[1,0,0],[-1,0,0],[0,1,0],[0,-1,0],[0,0,1],[0,0,-1]], dtype=np.int16)
pos = np.zeros((walks, 3), dtype=np.int16); returned = np.zeros(walks, bool)
for t in range(horizon):
    pos += steps[rng.integers(0, 6, walks)]
    returned |= ~pos.any(axis=1)
print(f"simulated P(return before time {horizon}) = {returned.mean():.4f}")

$\sum u_n=1.516386$ and $F=0.340537$, Pólya's constant. The simulation is a little lower, because walks that return after time $2000$ are not counted: $P(\text{first return}>n)$ decays like $n^{-1/2}$ in three dimensions, so the deficit at $n=2000$ is about $0.01$. In dimension $2$ the integral grows like $\frac1\pi\log s$ without bound: recurrence, but a slow one.

### P4.6
**How to find it.** Compute the bucket counts with `np.bincount`, then $\chi^2=\sum_b(N_b-n/m)^2/(n/m)$ and its $\chi^2_{m-1}$ $p$-value. For $m=1024$ and keys multiples of $64$, $k\bmod1024$ only takes the $16$ values $0,64,\dots,960$.

In [ ]:
m, n = 1024, 10**5
keys = 64 * rng.choice(10**8, size=n, replace=False)
A = (math.sqrt(5) - 1) / 2
hashes = {
    "k mod 1024": keys % 1024,
    "k mod 1021 (prime)": keys % 1021,
    "multiplicative": np.floor(m * np.mod(keys * A, 1.0)).astype(np.int64),
    "Python hash(str(k)) mod 1024": np.array([hash(str(int(k))) % 1024 for k in keys]),
}
for name, h in hashes.items():
    buckets = 1021 if "1021" in name else m
    c = np.bincount(h, minlength=buckets)
    e = n / buckets
    x = np.sum((c - e)**2 / e)
    print(f"{name:30s} chi2 = {x:12.1f}  (df {buckets-1}),  p = {stats.chi2.sf(x, buckets - 1):.3g},  longest chain {c.max()}")

Hashing by $k\bmod 1024$ uses $16$ buckets out of $1024$: $\chi^2$ is enormous and chains are $64$ times too long. The test catches it because the counts are wildly non-multinomial. A prime modulus, multiplicative hashing and Python's string hash all pass. (For keys that are floats, the products $kA$ lose precision for large $k$; integer multiplicative hashing, "Fibonacci hashing" with a $64$-bit multiplier, is what one uses in practice.)

Can a bad hash pass? Yes: $h(k)=$ (a counter incremented at each insertion) $\bmod m$ fills the buckets perfectly evenly, so $\chi^2$ is $0$, far too *small*, which is itself suspicious: a $\chi^2_{1023}$ variable is almost never below $900$. And a hash can be uniform on random keys and terrible on adversarial ones, which is what hash-flooding attacks exploit, and why Python randomises its string hash per process. The $\chi^2$ test checks one property, uniformity of the counts, through the multivariate central limit theorem; it says nothing about the rest.